# ALTE Common Corpus SIG — Stage 5 Colab Runner

This notebook connects **GitHub**, **Google Drive**, and **Colab** for the ALTE Common Corpus SIG / European CEFR Vocabulary Atlas pilot.

It first checks that the repository, Drive folders, taxonomy file, and English sample file are visible from Colab.

## Important methodological note

All LLM-generated outputs in this project are provisional Tier 4 candidate material. They are not validated CEFR data and require expert review.


## 1. Import basic libraries

In [ ]:
from pathlib import Path
from getpass import getpass
import os
import pandas as pd

print("Basic libraries loaded.")


## 2. Clone the GitHub repository

Run this if the repo has not already been cloned in the current Colab runtime.


In [ ]:
REPO_URL = "https://github.com/Pertam/ALTE-Common-Corpus-SIG.git"
REPO_DIR = Path("/content/ALTE-Common-Corpus-SIG")

if REPO_DIR.exists():
    print(f"Repo folder already exists: {REPO_DIR}")
else:
    !git clone https://github.com/Pertam/ALTE-Common-Corpus-SIG.git /content/ALTE-Common-Corpus-SIG

%cd /content/ALTE-Common-Corpus-SIG
!git status


## 3. Install requirements

In [ ]:
requirements = Path("/content/ALTE-Common-Corpus-SIG/requirements.txt")

if requirements.exists():
    !pip install -r /content/ALTE-Common-Corpus-SIG/requirements.txt
else:
    raise FileNotFoundError("requirements.txt not found in the repo. Add it to GitHub first.")


## 4. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 5. Define project paths

Edit `DRIVE_ROOT` only if you used a different Google Drive folder name.


In [ ]:
REPO_ROOT = Path("/content/ALTE-Common-Corpus-SIG")
DRIVE_ROOT = Path("/content/drive/MyDrive/ALTE-Common-Corpus-SIG")

TAXONOMY_PATH = REPO_ROOT / "taxonomy" / "cefr_function_taxonomy.csv"

EN_SAMPLE_PATH = (
    DRIVE_ROOT
    / "data"
    / "interim"
    / "en"
    / "samples"
    / "stage5_en_random_15_lemmas_all_sentences.csv"
)

EN_PASS1_OUTPUT = (
    DRIVE_ROOT
    / "data"
    / "outputs"
    / "en"
    / "pass1"
    / "en_function_tags_pass1.csv"
)

print("REPO_ROOT:", REPO_ROOT)
print("DRIVE_ROOT:", DRIVE_ROOT)
print("TAXONOMY_PATH:", TAXONOMY_PATH)
print("EN_SAMPLE_PATH:", EN_SAMPLE_PATH)
print("EN_PASS1_OUTPUT:", EN_PASS1_OUTPUT)


## 6. Check expected folders

In [ ]:
expected_paths = [
    REPO_ROOT,
    REPO_ROOT / "taxonomy",
    REPO_ROOT / "prompts",
    REPO_ROOT / "scripts",
    REPO_ROOT / "config",
    DRIVE_ROOT,
    DRIVE_ROOT / "data" / "interim" / "en" / "samples",
    DRIVE_ROOT / "data" / "outputs" / "en" / "pass1",
    DRIVE_ROOT / "data" / "outputs" / "en" / "pass2",
    DRIVE_ROOT / "data" / "outputs" / "en" / "pass3",
    DRIVE_ROOT / "data" / "outputs" / "en" / "qa",
    DRIVE_ROOT / "data" / "outputs" / "en" / "review_workbooks",
]

for path in expected_paths:
    print(f"{str(path):100} -> {path.exists()}")


## 7. Check that the taxonomy and English sample exist

The taxonomy should live in GitHub:

`taxonomy/cefr_function_taxonomy.csv`

The English sample should live in Drive:

`data/interim/en/samples/stage5_en_random_15_lemmas_all_sentences.csv`


In [ ]:
print("Taxonomy exists:", TAXONOMY_PATH.exists())
print("English sample exists:", EN_SAMPLE_PATH.exists())

if not TAXONOMY_PATH.exists():
    print("\nACTION NEEDED: Upload/rename the taxonomy file in GitHub as:")
    print("taxonomy/cefr_function_taxonomy.csv")

if not EN_SAMPLE_PATH.exists():
    print("\nACTION NEEDED: Put the English sample CSV in Google Drive at:")
    print(EN_SAMPLE_PATH)


## 8. Read and inspect the files

In [ ]:
if TAXONOMY_PATH.exists() and EN_SAMPLE_PATH.exists():
    taxonomy = pd.read_csv(TAXONOMY_PATH)
    sample = pd.read_csv(EN_SAMPLE_PATH)

    print("Taxonomy rows:", len(taxonomy))
    print("Taxonomy columns:", list(taxonomy.columns))
    print("\nSample rows:", len(sample))
    print("Sample columns:", list(sample.columns))

    display(sample.head())
else:
    print("Cannot read files yet. Fix the missing file path(s) above first.")


## 9. Optional: enter OpenAI API key for this session

Do not store the API key in GitHub.

Only run this when you are ready to call the OpenAI API.


In [ ]:
# Uncomment when ready to run API-based tagging.
# os.environ["OPENAI_API_KEY"] = getpass("Paste your OpenAI API key: ")
# print("OpenAI API key set for this Colab session.")


## 10. Next step

Once the path checks pass, add and run:

`scripts/00_validate_inputs.py`

Then proceed to the Stage 5 tagging pipeline:

1. Pass 1 sentence-level function tagging
2. Pass 2 blind validation
3. Pass 3 lemma-level aggregation
4. QA checks
5. Review workbook export
